In [ ]:
import matplotlib as pl
import matplotlib.pyplot as plt
import pandas as pd
import sklearn as sc
import scipy as sp
import numpy as np
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.cluster import DBSCAN
from sklearn.metrics import pairwise_distances




# Aufgabe B2
## Teil 1

In [ ]:
# Team A: Einführung der Daten
daten = pd.DataFrame(
    {
        "x": [12, 6, 4, 1, 0, 3, 6],
        "y": [2, 1, 10, 6, 3, 0, 12],
    },
    index=[f"P{i}" for i in range(7)],
)

display(daten)

#### a)

In [ ]:
# Paarweise euklidische Distanzmatrix
pairwise_distance = pairwise_distances(daten[["x", "y"]], metric="euclidean")
distanzmatrix = pd.DataFrame(pairwise_distance, index=daten.index, columns=daten.index)

# Kleinste Distanz ohne Hauptdiagonale bestimmen
distanz_ohne_diagonale = pairwise_distance.copy()
np.fill_diagonal(distanz_ohne_diagonale, np.inf)
i, j = np.unravel_index(np.argmin(distanz_ohne_diagonale), distanz_ohne_diagonale.shape)

punkt_1 = daten.index[i]
punkt_2 = daten.index[j]
min_distanz = pairwise_distance[i, j]

display(
    distanzmatrix.style
    .format("{:.2f}")
    .background_gradient(cmap="Blues")
    .set_caption("Paarweise euklidische Distanzmatrix")
)

ergebnis = pd.DataFrame(
    {
        "Punkt 1": [punkt_1],
        "Punkt 2": [punkt_2],
        "Distanz": [round(min_distanz, 4)],
    }
)

display(ergebnis.style.hide(axis="index").set_caption("Nächstes Punktepaar"))

#### b)

In [ ]:
# Agglomeratives Clustering mit drei Linkage-Verfahren
linkage_verfahren = {
    "Single Linkage": "single",
    "Complete Linkage": "complete",
    "Centroid Linkage": "centroid",
}

linkage_matrizen = {}

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))

for ax, (titel, methode) in zip(axes, linkage_verfahren.items()):
    Z = linkage(daten[["x", "y"]], method=methode, metric="euclidean")
    linkage_matrizen[methode] = Z

    dendrogram(
        Z,
        labels=daten.index.tolist(),
        ax=ax,
        color_threshold=0,
        above_threshold_color="black",
    )
    ax.set_title(titel, fontweight="bold")
    ax.set_xlabel("Datenpunkte")
    ax.set_ylabel("Euklidische Distanz")
    ax.grid(axis="y", alpha=0.25)

fig.suptitle("Vollständige Dendrogramme des agglomerativen Clusterings", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Vergleich der Verfahren bei einer sinnvollen Interpretation mit drei Gruppen.
# In b) ist keine feste Clusteranzahl vorgegeben; die drei Gruppen entsprechen der
# sichtbaren Struktur aus den Dendrogrammen: zwei Punktgruppen und P0 als Einzelgruppe.
vergleich = []

for titel, methode in linkage_verfahren.items():
    cluster_labels = fcluster(linkage_matrizen[methode], t=3, criterion="maxclust")
    cluster_text = []

    for cluster_id in sorted(set(cluster_labels)):
        punkte = daten.index[cluster_labels == cluster_id].tolist()
        cluster_text.append(f"C{cluster_id}: " + ", ".join(punkte))

    vergleich.append(
        {
            "Verfahren": titel,
            "3-Gruppen-Aufteilung": " | ".join(cluster_text),
            "Nächste Fusion nach Schnitt": round(linkage_matrizen[methode][-2, 2], 4),
        }
    )

vergleich_df = pd.DataFrame(vergleich)
display(vergleich_df.style.hide(axis="index").set_caption("Vergleich der agglomerativen Verfahren"))

**Auswahl des besten Clusterings:**  
Als bestes Clustering wird **Complete Linkage** gewählt. In Aufgabe b) ist keine feste Anzahl an Clustern vorgegeben, deshalb wird das Dendrogramm dort interpretiert, wo die Struktur am sinnvollsten ist. Bei diesem Datensatz sind drei Gruppen plausibel: `P1, P3, P4, P5`, `P2, P6` und `P0` als einzelne, weit entfernte Gruppe. **Single Linkage** ist schwach, weil es durch den Ketteneffekt getrennte Bereiche schon bei relativ kleiner Distanz verbindet. **Centroid Linkage** ist besser, kann aber durch Schwerpunktbildung ebenfalls früh größere Gruppen zusammenziehen. **Complete Linkage** bewertet die maximale Distanz zwischen Clustern und trennt die Gruppen am stabilsten; die nächste Fusion nach der 3-Gruppen-Struktur erfolgt erst bei einer deutlich höheren Distanz. Deshalb ist Complete Linkage für diesen Datensatz die beste Wahl.

#### c)

In [ ]:
# DBSCAN: kurze Parametersuche für genau zwei gültige Cluster.
# Ausreißer mit Label -1 zählen dabei nicht als Cluster.
parameter_test = []

for min_pts_test in [2, 3, 4]:
    for eps_test in [3.0, 3.5, 4.0, 4.25, 4.5, 5.0, 5.5, 6.0]:
        modell = DBSCAN(eps=eps_test, min_samples=min_pts_test)
        labels_test = modell.fit_predict(daten[["x", "y"]])
        cluster_anzahl = len(set(labels_test) - {-1})
        noise_anzahl = list(labels_test).count(-1)

        parameter_test.append(
            {
                "eps": eps_test,
                "MinPts": min_pts_test,
                "gültige Cluster": cluster_anzahl,
                "Geräuschpunkte": noise_anzahl,
                "Labels": ", ".join(map(str, labels_test)),
            }
        )

parameter_test_df = pd.DataFrame(parameter_test)
display(
    parameter_test_df[parameter_test_df["gültige Cluster"] == 2]
    .style
    .hide(axis="index")
    .set_caption("DBSCAN-Parameterkombinationen mit genau zwei gültigen Clustern")
)

In [ ]:
# Gewählte Parameter aus der Parametersuche
eps = 4.25
min_pts = 2

dbscan = DBSCAN(eps=eps, min_samples=min_pts)
dbscan_labels = dbscan.fit_predict(daten[["x", "y"]])
core_indices = set(dbscan.core_sample_indices_)

dbscan_ergebnis = daten.copy()
dbscan_ergebnis["Cluster"] = ["Ausreißer" if label == -1 else f"Cluster {label}" for label in dbscan_labels]
dbscan_ergebnis["Instanztyp"] = [
    "Geräuschinstanz" if label == -1 else "Kerninstanz" if i in core_indices else "Grenzinstanz"
    for i, label in enumerate(dbscan_labels)
]

anzahl_cluster = len(set(dbscan_labels) - {-1})
kerninstanzen = dbscan_ergebnis[dbscan_ergebnis["Instanztyp"] == "Kerninstanz"].index.tolist()
grenzinstanzen = dbscan_ergebnis[dbscan_ergebnis["Instanztyp"] == "Grenzinstanz"].index.tolist()
geraeuschinstanzen = dbscan_ergebnis[dbscan_ergebnis["Instanztyp"] == "Geräuschinstanz"].index.tolist()

print(f"Gewählte Parameter: eps = {eps}, MinPts = {min_pts}")
print(f"Anzahl gültiger Cluster: {anzahl_cluster}")

display(
    dbscan_ergebnis
    .style
    .set_caption("DBSCAN-Ergebnis und Instanztypen")
)

zusammenfassung = pd.DataFrame(
    {
        "Typ": ["Kerninstanzen", "Grenzinstanzen", "Geräuschinstanzen"],
        "Punkte": [
            ", ".join(kerninstanzen) if kerninstanzen else "keine",
            ", ".join(grenzinstanzen) if grenzinstanzen else "keine",
            ", ".join(geraeuschinstanzen) if geraeuschinstanzen else "keine",
        ],
    }
)

display(zusammenfassung.style.hide(axis="index").set_caption("Kern-, Grenz- und Geräuschinstanzen"))

##### Visualisierung des DBSCAN-Ergebnisses

In [ ]:
farben = {"Cluster 0": "tab:blue", "Cluster 1": "tab:orange", "Ausreißer": "tab:gray"}
marker = {"Kerninstanz": "D", "Grenzinstanz": "o", "Geräuschinstanz": "x"}

fig, ax = plt.subplots(figsize=(7.5, 5.5))

for punkt, zeile in dbscan_ergebnis.iterrows():
    ax.scatter(
        zeile["x"],
        zeile["y"],
        s=150,
        color=farben[zeile["Cluster"]],
        marker=marker[zeile["Instanztyp"]],
        edgecolor="black" if zeile["Instanztyp"] != "Geräuschinstanz" else None,
        linewidth=0.8,
        label=f'{zeile["Cluster"]} - {zeile["Instanztyp"]}',
    )
    ax.annotate(punkt, (zeile["x"], zeile["y"]), xytext=(7, 5), textcoords="offset points")

# Doppelte Legendeneinträge entfernen
handles, labels_legend = ax.get_legend_handles_labels()
eindeutig = dict(zip(labels_legend, handles))

ax.set_title(f"DBSCAN mit eps={eps}, MinPts={min_pts}", fontweight="bold")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(alpha=0.3)
ax.legend(eindeutig.values(), eindeutig.keys(), loc="best")

plt.tight_layout()
plt.show()

**Auswertung DBSCAN:**  
Mit `eps = 4.25` und `MinPts = 2` entstehen genau zwei gültige Cluster. `P0` wird als Geräuschinstanz erkannt, weil der Punkt zu weit von den anderen Punkten entfernt liegt. Die Cluster bestehen aus `P1, P3, P4, P5` sowie `P2, P6`. Alle Punkte in den beiden Clustern sind Kerninstanzen; Grenzinstanzen gibt es bei dieser Parametereinstellung nicht.

#### d)

In [ ]:
# Distanzmatrix nach dem DBSCAN-Clustering sortieren.
# Cluster 0 und Cluster 1 kommen zuerst, der Ausreißer P0 steht am Ende.
sortier_info = dbscan_ergebnis.copy()
sortier_info["Sortierung"] = dbscan_labels
sortier_info["Sortierung"] = sortier_info["Sortierung"].replace(-1, 99)
sortier_info["Punktnummer"] = [int(punkt[1:]) for punkt in sortier_info.index]
sortier_info = sortier_info.sort_values(["Sortierung", "Punktnummer"])

sortierte_punkte = sortier_info.index.tolist()
daten_sortiert = daten.loc[sortierte_punkte]

distanzmatrix_sortiert = pairwise_distances(daten_sortiert[["x", "y"]], metric="euclidean")
heatmap_labels = [f"{punkt} ({dbscan_ergebnis.loc[punkt, 'Cluster']})" for punkt in sortierte_punkte]

distanzmatrix_sortiert_df = pd.DataFrame(
    distanzmatrix_sortiert,
    index=heatmap_labels,
    columns=heatmap_labels,
)

print("Sortierte Reihenfolge der Punkte:")
display(sortier_info[["x", "y", "Cluster", "Instanztyp"]])

plt.figure(figsize=(9, 7))
sns.heatmap(
    distanzmatrix_sortiert_df,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={"label": "Euklidische Distanz"},
)

plt.title("Heatmap der Distanzmatrix nach DBSCAN-Clustern", fontweight="bold")
plt.xlabel("Sortierte Datenpunkte")
plt.ylabel("Sortierte Datenpunkte")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

**Interpretation der Heatmap:**  
Die Heatmap zeigt nach der Sortierung zwei erkennbare Clusterbereiche. `Cluster 1` mit `P2` und `P6` ist sehr kompakt, da die Distanz zwischen diesen beiden Punkten nur `2.83` beträgt. `Cluster 0` mit `P1, P3, P4, P5` ist weniger kompakt, aber durch nahe Nachbarschaften zusammenhängend: `P1` liegt nah bei `P5`, `P5` nah bei `P4` und `P4` nah bei `P3`. Genau diese Dichteverkettung passt zu DBSCAN. `P0` hat zu allen anderen Punkten deutlich größere Distanzen und wird deshalb sinnvoll als Geräuschinstanz erkannt. Insgesamt ist das DBSCAN-Clustering für diesen Datensatz plausibel und gut interpretierbar, aber nicht perfekt kompakt wie ein ideales kugelförmiges Clustering.

## Teil 2

#### a)

#### b)

#### c)

#### d)

#### e)

#### f)

#### g)

#### h)

#### i)

#### j)

#### k)

#### l)

#### m)